In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [4]:
df_ch_przeplyw = pd.read_parquet('../../data/processed/Chorzów/przeplyw.parquet')
df_ch_scieki = pd.read_parquet('../../data/processed/Chorzów/scieki.parquet')
df_ch_awarie = pd.read_parquet('../../data/processed/Chorzów/awarie.parquet')
df_ch_energia = pd.read_parquet('../../data/processed/Chorzów/energia.parquet')
df_ch_opady = pd.read_parquet('../../data/processed/Chorzów/suma_opadow.parquet')

In [6]:
df_ch_energia

,Data,Czas,Zuzycie_energii_elektrycznej,Sposób_pomiaru,Data_czas
0,13.10.2024,00:00:00,515940,Wartość z pomiaru,2024-10-13 00:00:00
1,13.10.2024,00:00:54,515940,Wartość z pomiaru,2024-10-13 00:00:54
2,13.10.2024,00:01:06,515941,Wartość z pomiaru,2024-10-13 00:01:06
3,13.10.2024,00:01:14,515941,Wartość z pomiaru,2024-10-13 00:01:14
4,13.10.2024,00:01:18,515941,Wartość z pomiaru,2024-10-13 00:01:18
...,...,...,...,...,...
664016,12.10.2026,23:57:55,587592,Wartość z pomiaru,2026-10-12 23:57:55
664017,12.10.2026,23:58:03,587593,Wartość z pomiaru,2026-10-12 23:58:03
664018,12.10.2026,23:58:17,587593,Wartość z pomiaru,2026-10-12 23:58:17
664019,12.10.2026,23:58:35,587593,Wartość z pomiaru,2026-10-12 23:58:35


In [8]:
df = df_ch_energia.copy()

df["Sposób_pomiaru"] = df["Sposób_pomiaru"].astype(str).str.strip()
df = df[df["Sposób_pomiaru"] == "Wartość z pomiaru"].copy()

if "Data_czas" in df.columns:
    df["Data_czas"] = pd.to_datetime(df["Data_czas"])
    df = df.set_index("Data_czas")

df = df.rename(columns={"Zuzycie_energii_elektrycznej": "energia"})

# Konwersja na wartości liczbowe
df["energia"] = pd.to_numeric(
    df["energia"].astype(str).str.replace(",", ".", regex=False),
    errors="coerce"
)

df = df[df["energia"].notna()]
df = df.sort_index()

print(df[["energia"]].head(10))

                     energia
Data_czas                   
2024-10-13 00:00:00   515940
2024-10-13 00:00:54   515940
2024-10-13 00:01:06   515941
2024-10-13 00:01:14   515941
2024-10-13 00:01:18   515941
2024-10-13 00:01:26   515941
2024-10-13 00:01:34   515941
2024-10-13 00:01:52   515941
2024-10-13 00:05:00   515941
2024-10-13 00:10:00   515941


In [9]:
# Usuwamy zduplikowane daty, zachowując największy odczyt licznika energii
df = df.groupby(level=0).agg({"energia": "max"})
df = df.sort_index()

df.head()

,energia
Data_czas,
2024-10-13 00:00:00,515940
2024-10-13 00:00:54,515940
2024-10-13 00:01:06,515941
2024-10-13 00:01:14,515941
2024-10-13 00:01:18,515941


In [13]:
#Godzinowe zużycie energii z licznika narastającego
hourly = df["energia"].resample("1h").last()

df_1h = pd.DataFrame({
    "zuzycie_energii": hourly.diff()
})

# Reset licznika lub nieprawidłowy odczyt
df_1h.loc[df_1h["zuzycie_energii"] < 0, "zuzycie_energii"] = np.nan

df_lgbm = df_1h.copy()

print(df_lgbm.head(10))

                     zuzycie_energii
Data_czas                           
2024-10-13 00:00:00              NaN
2024-10-13 01:00:00              3.0
2024-10-13 02:00:00              2.0
2024-10-13 03:00:00              3.0
2024-10-13 04:00:00              2.0
2024-10-13 05:00:00              3.0
2024-10-13 06:00:00              2.0
2024-10-13 07:00:00              3.0
2024-10-13 08:00:00              3.0
2024-10-13 09:00:00              4.0
